# 02: Advanced Outlier Detection & Statistical Bounding

Outliers can drastically warp decision boundaries. We will use Violin Plots (which show both boxplot statistics and KDE density) and calculate strict IQR/Z-Score bounds.

In [ ]:
import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt
from scipy import stats

train = pd.read_csv('../../data/train.csv')
num_cols = train.select_dtypes(include=np.number).columns.tolist()
if 'id' in num_cols: num_cols.remove('id')
print(f'Analyzing {len(num_cols)} numerical columns.')

## 1. Distribution & Outlier Visualization (Violin + Swarm)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 18))
axes = axes.flatten()

for i, col in enumerate(num_cols[:6]):
    sns.violinplot(y=train[col], ax=axes[i], color='skyblue', inner='quartile')
    axes[i].set_title(f'Violin Plot of {col}', fontsize=14, fontweight='bold')
    axes[i].set_ylabel('')
    
plt.tight_layout()
plt.show()

## 2. Statistical Outlier Boundaries (IQR Method)
We will compute the strict upper and lower bounds for our features.

In [ ]:
outlier_stats = []
for col in num_cols:
    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers_count = ((train[col] < lower) | (train[col] > upper)).sum()
    outlier_stats.append({'Feature': col, 'Q1': Q1, 'Q3': Q3, 'Lower Bound': lower, 'Upper Bound': upper, 'Outliers': outliers_count})

stats_df = pd.DataFrame(outlier_stats)
display(stats_df.style.background_gradient(subset=['Outliers'], cmap='Reds'))

## 3. Bivariate Outlier Detection
Are there outliers that only appear when we consider the target variable?

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='health_condition', y='sleep_duration', data=train, palette='Set2')
plt.title('Sleep Duration Outliers Segmented by Health Condition', fontsize=16)
plt.show()